# Unidad 2 · Comparar dos medias

Reproduce el ejemplo resuelto (neumáticos de una flota, diseño independiente y pareado) y la pieza del
caso integrador (pulpo 6 contra pulpo 4). Lee los datos de `../datos/`.

In [ ]:
from pathlib import Path
from math import sqrt
import numpy as np
import pandas as pd
from scipy import stats

DATOS = Path("..") / "datos"
RES = {}

def welch_gl(x, y):
    a, b = x.var(ddof=1) / len(x), y.var(ddof=1) / len(y)
    return (a + b) ** 2 / (a ** 2 / (len(x) - 1) + b ** 2 / (len(y) - 1))

def cohen_d(x, y):
    n1, n2 = len(x), len(y)
    sp = sqrt(((n1 - 1) * x.var(ddof=1) + (n2 - 1) * y.var(ddof=1)) / (n1 + n2 - 2))
    return (x.mean() - y.mean()) / sp

## 1. Ejemplo · diseño independiente (16 camiones, 8 con cada marca)

In [ ]:
df = pd.read_csv(DATOS / "u02_neumaticos.csv")
ind = df[df.diseno == "independiente"]
A = ind[ind.marca == "A"].consumo_l100km.to_numpy()
B = ind[ind.marca == "B"].consumo_l100km.to_numpy()
print("Shapiro A:", stats.shapiro(A).pvalue, " Shapiro B:", stats.shapiro(B).pvalue)
comb = stats.ttest_ind(A, B, equal_var=True)
welch = stats.ttest_ind(A, B, equal_var=False)
print("t combinada:", comb)
print("Welch:", welch, " gl =", welch_gl(A, B))
print("d de Cohen:", cohen_d(A, B))
print("Mann-Whitney:", stats.mannwhitneyu(A, B))
RES.update(ej_t_comb=comb.statistic, ej_p_comb=comb.pvalue, ej_t_welch=welch.statistic, ej_p_welch=welch.pvalue)

## 2. Ejemplo · diseño pareado (8 camiones con las dos marcas)

In [ ]:
par = df[df.diseno == "pareado"].pivot(index="camion", columns="marca", values="consumo_l100km")
d = par.A - par.B
print(d.describe())
r = stats.ttest_rel(par.A, par.B)
print("t pareada:", r, " d_z =", d.mean() / d.std(ddof=1))
print("Mismos datos como si fueran independientes:", stats.ttest_ind(par.A, par.B, equal_var=False))
print("Wilcoxon de rangos con signo:", stats.wilcoxon(par.A, par.B))
RES.update(ej_t_par=r.statistic, ej_p_par=r.pvalue)

## 3. Caso · ¿el pulpo 6 y el pulpo 4 desplazan distinto el registro?

$H_0: \mu_6 = \mu_4$ contra $H_1: \mu_6 \ne \mu_4$, con $\alpha = 0{,}05$.

In [ ]:
reg = pd.read_csv(DATOS / "registro.csv")
a = reg[reg.maquina == "pulpo 6"].desplazamiento_mm.to_numpy()
b = reg[reg.maquina == "pulpo 4"].desplazamiento_mm.to_numpy()
print(reg.groupby("maquina").desplazamiento_mm.agg(["count", "mean", "std"]))
print("Shapiro:", stats.shapiro(a).pvalue, stats.shapiro(b).pvalue)
w = stats.ttest_ind(a, b, equal_var=False)
c = stats.ttest_ind(a, b, equal_var=True)
gl = welch_gl(a, b)
print(f"Welch: t = {w.statistic:.2f}, gl = {gl:.1f}, p = {w.pvalue:.4f}")
print(f"Combinada: t = {c.statistic:.2f}, gl = {len(a) + len(b) - 2}, p = {c.pvalue:.4f}")
print("IC 95 % de la diferencia (Welch):", w.confidence_interval(0.95))
dd = cohen_d(a, b)
mw = stats.mannwhitneyu(a, b)
print(f"d de Cohen = {dd:.2f}   Mann-Whitney U = {mw.statistic:.0f}, p = {mw.pvalue:.4f}")
RES.update(caso_t_welch=w.statistic, caso_p_welch=w.pvalue, caso_gl_welch=gl, caso_t_comb=c.statistic,
           caso_p_comb=c.pvalue, caso_d=dd, caso_p_mw=mw.pvalue)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 3))
ax.boxplot([a, b]); ax.set_xticks([1, 2], ["pulpo 6", "pulpo 4"])
ax.axhline(0.10, ls="--", color="gray")
ax.set_ylabel("desplazamiento (mm)")
plt.show()